# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema and referencing all entities using their `@id` identifiers.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as an object)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

- **RecordSet** corresponds to a table or structured collection of records in the Croissant schema.
- **Fields** and **columns** correspond to the variables (columns) of each record set.

Let's enumerate all available record sets and their fields using their `@id` fields.

In [ ]:
# List all available RecordSets by their @id and names
print("Available RecordSets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"@id: {rs['@id']} | type: {rs.get('@type', '')} | name: {rs.get('name', '')}")

# Show fields (columns) for each RecordSet by @id
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        if isinstance(f, dict):
            print(f"  - Field @id: {f.get('@id')} | name: {f.get('name','')} | type: {f.get('@type','')}")
        else:
            print(f"  - Field @id: {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set `@id` and field `@id`s from the overview below. If unsure, use the printed output above to update which record set/fields you want to extract.

In [ ]:
# Here, we pick the primary data RecordSet. Replace the value below with the specific `@id` of the record set you wish to use (from overview output above).
# Example: data_recordset_id = "cr:recordSet/second_primary_crc_records"

# Gather all available RecordSet @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("RecordSet @ids for extraction:")
for rs_id in record_set_ids:
    print(rs_id)
# Pick the main data record set (for this dataset, often only one table)
data_recordset_id = record_set_ids[0]  # Update if there are multiple - use the correct @id

dataframes = {}
for rs_id in record_set_ids:
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"\nRecordSet @id: {rs_id} | Loaded shape: {dataframes[rs_id].shape}")
        print(f"Columns (@id): {list(dataframes[rs_id].columns)[:10]}{' ...' if len(dataframes[rs_id].columns) > 10 else ''}")
        print(dataframes[rs_id].head(2))


## 4. Exploratory Data Analysis (EDA)
We will select a numeric field from the DataFrame for analysis by using its column `@id`.

*Steps:*
- Filter records based on a numerical field (e.g., Age).
- Normalize this field.
- Group by a categorical field (e.g., Sex) if available.

Be sure to use the exact `@id` as the DataFrame column name.

In [ ]:
# Explore structure to pick a numeric field.
main_df = dataframes[data_recordset_id]
print("Columns available (use @id):")
for col in main_df.columns:
    print(col)

# For this example, suppose '@id' for Age is 'cr:field/Age' and for Sex is 'cr:field/Sex'
# Replace with the actual @ids from your schema if they differ
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    if 'Age' in col or 'age' in col:
        numeric_field_id = col
    if 'Sex' in col or 'Gender' in col or 'sex' in col:
        group_field_id = col
print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

if numeric_field_id is not None:
    # Convert numeric field to float if necessary
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean()  # Use mean as an example threshold
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.1f} (mean): {len(filtered_df)} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by sex/gender, if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_age')
        print("\nGrouped by group_field ({}) - mean of {}:".format(group_field_id, numeric_field_id))
        print(grouped_df)
else:
    print("No numeric field (e.g. Age) was located in columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its grouping by a categorical field.

We'll create a histogram for the Age distribution and, if available, a boxplot grouped by Sex (or equivalent).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    main_df[numeric_field_id].dropna().hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel("Value")
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion

- We successfully loaded the FAIR^2 dataset using its Croissant schema and the `mlcroissant` library.
- By referencing all entities by their `@id`, we explored tables (record sets), variables (fields), and extracted the primary dataset for analysis.
- Using EDA, we demonstrated filtering, normalization, grouping, and visualization of a numeric field (e.g., Age), and explored group differences (e.g., by Sex).

Further analysis can include more advanced statistical testing, additional groupings, or machine learning workflows—all while referencing fields with their stable `@id`s.